<a href="https://colab.research.google.com/github/iamshekhark/BIS-LAB/blob/main/Cuckoo%20Search/BIS_Cuckoo_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import time
import math

# Objective function
def fitness(x):
    return np.sum(x ** 2)

# Levy flight calculation
def levy_flight():
    beta = 1.5
    sigma = (
        math.gamma(1 + beta) * np.sin(np.pi * beta / 2) /
        (math.gamma((1 + beta) / 2) * beta *
         2 ** ((beta - 1) / 2))
    ) ** (1 / beta)

    u = np.random.normal(0, sigma)
    v = np.random.normal(0, 1)

    return u / (abs(v) ** (1 / beta) + 1e-10)


# Cuckoo Search Algorithm
def cuckoo_search(nests):

    dim = 5
    max_iter = 500
    pa = 0.25
    lower = -10
    upper = 10

    # Initialize nests
    population = np.random.uniform(
        lower, upper, (nests, dim)
    )

    scores = np.array([fitness(x) for x in population])

    best_index = np.argmin(scores)
    best_nest = population[best_index].copy()
    best_fitness = scores[best_index]

    convergence_speed = None

    start_time = time.time()

    for iteration in range(max_iter):

        # Generate new cuckoo solutions
        for i in range(nests):

            new_nest = population[i].copy()

            step = np.array([
                levy_flight() for _ in range(dim)
            ])

            new_nest += 0.01 * step * (
                population[i] - best_nest
            )

            new_nest = np.clip(new_nest, lower, upper)

            new_fitness = fitness(new_nest)

            # Replace random nest if solution is better
            j = np.random.randint(nests)

            if new_fitness < scores[j]:
                population[j] = new_nest
                scores[j] = new_fitness

        # Abandon some nests and create new solutions
        abandon_count = max(1, int(pa * nests))

        worst_indices = np.argsort(scores)[-abandon_count:]

        for j in worst_indices:
            population[j] = np.random.uniform(
                lower, upper, dim
            )
            scores[j] = fitness(population[j])

        # Update best solution
        best_index = np.argmin(scores)

        if scores[best_index] < best_fitness:
            best_fitness = scores[best_index]
            best_nest = population[best_index].copy()

        # Record convergence speed
        if convergence_speed is None and best_fitness <= 1e-6:
            convergence_speed = iteration + 1

    end_time = time.time()

    execution_time = end_time - start_time

    return best_fitness, convergence_speed, execution_time


# Compare different nest populations
nest_sizes = [10, 20, 30, 50]

print("CUCKOO SEARCH OPTIMIZATION")
print("--------------------------")

print("Nests\tBest Fitness\tConvergence\tTime (sec)")

for n in nest_sizes:

    best, convergence, execution_time = cuckoo_search(n)

    print(
        n, "\t",
        "%.8f" % best, "\t",
        convergence if convergence is not None else "Not reached",
        "\t",
        "%.4f" % execution_time
    )


CUCKOO SEARCH OPTIMIZATION
--------------------------
Nests	Best Fitness	Convergence	Time (sec)
10 	 23.15632216 	 Not reached 	 0.4150
20 	 8.70320516 	 Not reached 	 0.7352
30 	 4.70461891 	 Not reached 	 1.1652
50 	 4.26014206 	 Not reached 	 4.0490
